

# Laboratorio: RAG (Retrieval Augmented Generation) con LlamaIndex

**Objetivo:** Aprender a construir un sistema RAG que permite hacer preguntas
sobre documentos específicos, usando el Informe ASG 2023 de Scotiabank Chile.

**Características:**
- Modelo local Qwen2.5-3B-Instruct (no requiere API key ni permisos)
- Embeddings multilingües para español
- Comparación: LLM sin RAG vs LLM con RAG
- Sección opcional: Re-parseo con LlamaParse

**¿Qué es RAG?**
RAG (Retrieval Augmented Generation) es una técnica que mejora las respuestas
de un LLM al proporcionarle contexto relevante extraído de documentos.
El flujo es: Query → Buscar documentos relevantes → Generar respuesta con contexto.

## 1. Setup e Instalación de Dependencias

In [ ]:
# Instalar dependencias
# Transformers y cuantización
!pip install -q transformers accelerate bitsandbytes
# LlamaIndex core y componentes
!pip install -q llama-index-core llama-index-llms-huggingface llama-index-embeddings-huggingface llama-index-readers-file
# Utilidades
!pip install -q sentence-transformers pypdf
print("✓ Dependencias instaladas")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 90.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.0/92.0 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 328.2/328.2 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 52.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.7/150.7 kB 10.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
✓ Dependencias instaladas


## 2. Verificar GPU disponible

In [ ]:
import torch
if torch.cuda.is_available():
    print(f"✓ GPU detectada: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM disponible: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    USE_GPU = True
else:
    print("⚠ No se detectó GPU. El modelo se ejecutará en CPU (será más lento)")
    print("  En Google Colab: Runtime > Change runtime type > GPU")
    USE_GPU = False

✓ GPU detectada: Tesla T4
  VRAM disponible: 14.7 GB


## 3. Descargar el Documento PDF

Usaremos el **Resumen ASG 2023 de Scotiabank Chile**, un documento que contiene
información sobre las iniciativas ambientales, sociales y de gobernanza del banco.

In [ ]:
import urllib.request
import os

# URL del documento
PDF_URL = "https://cdn.aglty.io/scotiabank-chile/scotiabankpdf/Resumen_ASG_2023.pdf"
PDF_PATH = "Resumen_ASG_2023.pdf"

# Descargar si no existe
if not os.path.exists(PDF_PATH):
    print("📥 Descargando documento PDF...")
    urllib.request.urlretrieve(PDF_URL, PDF_PATH)
    print(f"✓ Documento descargado: {PDF_PATH}")
else:
    print(f"✓ Documento ya existe: {PDF_PATH}")

# Verificar tamaño
file_size = os.path.getsize(PDF_PATH) / 1024
print(f"  Tamaño: {file_size:.1f} KB")

📥 Descargando documento PDF...
✓ Documento descargado: Resumen_ASG_2023.pdf
  Tamaño: 3358.6 KB


## 4. Cargar y Procesar el Documento

Usamos LlamaIndex para cargar el PDF y dividirlo en chunks (fragmentos)
que serán indexados para la búsqueda.

In [ ]:
from llama_index.core import SimpleDirectoryReader
from llama_index.core.node_parser import SentenceSplitter

print("📄 Cargando y procesando documento...")
# Cargar el PDF
documents = SimpleDirectoryReader(
    input_files=[PDF_PATH]
).load_data()
print(f"✓ Documento cargado: {len(documents)} página(s)")

📄 Cargando y procesando documento...
✓ Documento cargado: 24 página(s)


In [ ]:
# Dividir en chunks más pequeños para mejor retrieval
parser = SentenceSplitter(
    chunk_size=512,      # Tamaño de cada chunk en tokens
    chunk_overlap=50     # Overlap entre chunks para mantener contexto
)

nodes = parser.get_nodes_from_documents(documents)
print(f"✓ Documento dividido en {len(nodes)} chunks")

# Veamos un ejemplo de chunk
print("\n📝 Ejemplo de chunk:")
print("-" * 50)
print(nodes[0].text[:500] + "...")


✓ Documento dividido en 36 chunks

📝 Ejemplo de chunk:
--------------------------------------------------
Por un futuro sostenible
El compromiso ASG de Scotiabank Chile en 2023...


## 5. Configurar Embeddings para Búsqueda Semántica

Los embeddings convierten texto en vectores numéricos que capturan el significado.
Usamos un modelo multilingüe que funciona bien con español.

In [ ]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings

print("📚 Cargando modelo de embeddings...")
# Modelo de embeddings multilingüe (soporta español)
embed_model = HuggingFaceEmbedding(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    device="cuda" if USE_GPU else "cpu"
)

# Configurar como modelo global
Settings.embed_model = embed_model

print("✓ Modelo de embeddings cargado")

📚 Cargando modelo de embeddings...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✓ Modelo de embeddings cargado


In [ ]:
# Ejemplo: ver cómo se ve un embedding
ejemplo_embedding = embed_model.get_text_embedding("Scotiabank Chile")
print(f"\n📊 Dimensiones del embedding: {len(ejemplo_embedding)}")
print(f"   Primeros 5 valores: {ejemplo_embedding[:5]}")


📊 Dimensiones del embedding: 384
   Primeros 5 valores: [-0.004931570962071419, -0.09855782240629196, 0.03136477246880531, -0.013769697397947311, 0.08304782211780548]


## 6. Crear el Índice Vectorial (Vector Store)

El índice vectorial almacena los embeddings de todos los chunks
y permite buscar los más similares a una query.

In [ ]:
from llama_index.core import VectorStoreIndex

print("🔍 Creando índice vectorial...")
# Crear índice a partir de los nodos
index = VectorStoreIndex(nodes)
print("✓ Índice vectorial creado")
print(f"  Total de vectores indexados: {len(nodes)}")

🔍 Creando índice vectorial...
✓ Índice vectorial creado
  Total de vectores indexados: 36


## 7. 🧪 SECCIÓN 1: Probar el Retriever

**Objetivo:** Entender cómo funciona la búsqueda semántica antes de usar el LLM.

El retriever busca los chunks más relevantes para una query dada.

In [ ]:
# Crear retriever (buscador)
retriever = index.as_retriever(
    similarity_top_k=3  # Retornar los 3 chunks más relevantes
)
print("=" * 60)
print("🔎 PROBANDO EL RETRIEVER")
print("=" * 60)

🔎 PROBANDO EL RETRIEVER


In [ ]:
# Queries de ejemplo para probar
queries_prueba = [
    "¿Cuál es la huella de carbono de Scotiabank?",
    "¿Qué es el programa ScotiaInspira?",
    "¿Cuántas mujeres trabajan en el banco?",
]

In [ ]:
for query in queries_prueba:
    print(f"\n📝 Query: {query}")
    print("-" * 50)

    # Recuperar documentos relevantes
    retrieved_nodes = retriever.retrieve(query)

    for i, node in enumerate(retrieved_nodes, 1):
        print(f"\n🔹 Documento {i} (Score: {node.score:.4f}):")
        # Mostrar solo los primeros 300 caracteres
        texto = node.text[:300].replace('\n', ' ')
        print(f"   {texto}...")

    print("\n" + "=" * 60)


📝 Query: ¿Cuál es la huella de carbono de Scotiabank?
--------------------------------------------------

🔹 Documento 1 (Score: 0.6450):
   Scotiabank es una  organización reconocida  a nivel global por su  aporte a la lucha contra  el cambio climático y la  promoción de políticas  bancarias que garanticen  la preservación del  medio ambiente. En este  ámbito, el Banco opera a  partir de una Estrategia  Climática que establece  tanto me...

🔹 Documento 2 (Score: 0.5510):
   Huella de carbono y otros indicadores ambientales En 2023, con miras a cumplir nuestras metas y  compromisos ambientales, continuamos monito- reando las emisiones de Gases de Efecto Inverna- dero que genera la organización. De acuerdo con nuestro cálculo, elaborado sobre  la base del GHG Protocol, d...

🔹 Documento 3 (Score: 0.5456):
   Durante el último año, Scotiabank estableció nue- vas metas climáticas para la organización a nivel  global, que abordan las oportunidades y riesgos  relacionados con el clima a cor

### 💡 Ejercicio 1: Prueba tus propias queries

Modifica la celda de abajo para probar diferentes preguntas y ver qué documentos retorna el retriever.

In [ ]:
# TODO: Escribe tu propia query aquí
mi_query = "Directorio Paritario"

In [ ]:
retrieved = retriever.retrieve(mi_query)
for i, node in enumerate(retrieved, 1):
    print(f"\n🔹 Documento {i} (Score: {node.score:.4f}):")
    print(f"   {node.text[:400]}...")


🔹 Documento 1 (Score: 0.5032):
   Directorio paritario
Presidente
Chileno
Salvador 
Said
Vicepresidente
Chileno
Emilio        
Deik
Directora
Chilena
Karen     
Ergas
Directora
Chilena
Fernanda 
Vicente
Directora
Brasileña
Raquel  
Costa
Director
Chileno
Gonzalo  
Said
Director
Chileno
Francisco 
Matte
Directora 
Suplente
Canadiense
Thayde 
Olarte
En una muestra de nuestro compromiso con la creación de entornos laborales libres de...

🔹 Documento 2 (Score: 0.4827):
   61.723 KG
RESIDUOS NO PELIGROSOS RECICLADOS 
EN NUESTRO EDIFICIO CORPORATIVO.
REPRESENTÓ EMISIÓN DE NUESTRO 
PRIMER BONO VERDE.
EN TRANSACCIONES DE BONOS Y 
PRÉSTAMOS ASG.
MMUSD 35
MMUSD 31.000
7
ORGANIZACIONES SE 
ADJUDICARON LOS 
FONDOS CONCURSABLES 
DEL PROGRAMA 
SCOTIAINSPIRA EN 2023.
45.371
1.520
PERSONAS FUERON 
BENEFICIADAS CON 
NUESTRAS INICIATIVAS 
DE VOLUNTARIADO 
CORPORATIVO.
ESCOLARES ...

🔹 Documento 3 (Score: 0.4595):
   A su vez, en 
diciembre, se presentó al Directorio la gestión y avan-
ces ASG durante e

## 8. Cargar el Modelo LLM Base (Qwen2.5-3B)

Usamos Qwen2.5-3B-Instruct, un modelo de 3B parámetros que:
- No requiere API key ni permisos especiales
- Es suficientemente pequeño para correr en Colab con GPU T4
- Tiene buen rendimiento en español

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from llama_index.llms.huggingface import HuggingFaceLLM

print("🚀 Cargando modelo Qwen2.5-3B-Instruct...")

# Nombre del modelo
model_id = "Qwen/Qwen2.5-3B-Instruct"

# Configuración de cuantización 4-bit para ahorrar memoria
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

# Cargar tokenizer
print("  Cargando tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Cargar modelo
print("  Cargando modelo cuantizado (puede tomar 1-2 minutos)...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config if USE_GPU else None,
    device_map="auto" if USE_GPU else None,
    trust_remote_code=True,
    torch_dtype=torch.float16 if USE_GPU else torch.float32,
)

🚀 Cargando modelo Qwen2.5-3B-Instruct...
  Cargando tokenizer...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/3.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
print(f"✓ Modelo cargado")
if USE_GPU:
    print(f"  Memoria GPU usada: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

# Configurar LLM para LlamaIndex
llm = HuggingFaceLLM(
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    generate_kwargs={
        "temperature": 0.1,
        "top_p": 0.9,
        "do_sample": True,
    },
    device_map="auto" if USE_GPU else None,
)

# Configurar como LLM global
Settings.llm = llm
print("✓ LLM configurado en LlamaIndex")


✓ Modelo cargado
  Memoria GPU usada: 2.37 GB
✓ LLM configurado en LlamaIndex


## 9. 🧪 SECCIÓN 2: Probar el LLM Base SIN RAG

**Objetivo:** Ver las limitaciones del LLM cuando no tiene acceso al documento.

El modelo solo puede responder con su conocimiento de entrenamiento.


In [ ]:
# Preguntas específicas sobre el documento
preguntas_sin_rag = [
    "¿Cuál fue la huella de carbono de Scotiabank Chile en 2023?",
    "¿Qué es el programa ScotiaInspira y cuánto dinero entregó en 2023?",
    "¿Cómo está compuesto el directorio paritario de Scotiabank Chile?",
]

In [ ]:
for pregunta in preguntas_sin_rag:
    print(f"\n❓ Pregunta: {pregunta}")
    print("-" * 50)

    # Generar respuesta SIN contexto (el LLM solo usa su conocimiento)
    respuesta = llm.complete(pregunta)
    print(f"🤖 Respuesta (sin RAG):\n{respuesta.text}")
    print("\n" + "=" * 60)


❓ Pregunta: ¿Cuál fue la huella de carbono de Scotiabank Chile en 2023?
--------------------------------------------------
🤖 Respuesta (sin RAG):
 | Scotiabank Chile
Inicio / Noticias / Cuál fue la huella de carbono de Scotiabank Chile en 2023?
La entidad financiera se comprometió a reducir su huella de carbono en un 50% para 2030, y ha logrado reducir su huella de carbono en un 17% desde 2019.
En 2023, la huella de carbono de Scotiabank Chile se redujo en un 17% con respecto al año anterior. La entidad financiera se comprometió a reducir su huella de carbono en un 50% para 2030, lo que significa que aún queda un camino por recorrer para alcanzar el objetivo.
El informe de la huella de carbono de Scotiabank Chile 2023, presentado por la entidad financiera, muestra que la huella de carbono de la empresa se redujo en un 17% en 2023, lo que representa una reducción de 14,684 toneladas de CO2 equivalente. Esto es un resultado significativo, ya que la empresa se comprometió a reducir su hu

### 💡 Observaciones

Notarás que el LLM:
- Puede inventar información (alucinaciones)
- No conoce datos específicos del documento de 2023
- Da respuestas genéricas o incorrectas

**Esto demuestra por qué necesitamos RAG.**

## 10. 🧪 SECCIÓN 3: Probar el RAG Completo

**Objetivo:** Ver cómo mejora la calidad de las respuestas al agregar contexto del documento.

In [ ]:
# Crear query engine (combina retriever + LLM)
query_engine = index.as_query_engine(
    similarity_top_k=3,  # Usar top 3 chunks más relevantes
)

In [ ]:
# Las mismas preguntas de antes
preguntas_con_rag = [
    "¿Cuál fue la huella de carbono de Scotiabank Chile en 2023?",
    "¿Qué es el programa ScotiaInspira y cuánto dinero entregó en 2023?",
    "¿Cómo está compuesto el directorio paritario de Scotiabank Chile?",
]

In [ ]:
for pregunta in preguntas_con_rag:
    print(f"\n❓ Pregunta: {pregunta}")
    print("-" * 50)

    # Generar respuesta CON contexto del documento
    respuesta = query_engine.query(pregunta)
    print(f"📚 Respuesta (con RAG):\n{respuesta.response}")

    # Mostrar las fuentes usadas
    print("\n📎 Fuentes utilizadas:")
    for i, source in enumerate(respuesta.source_nodes, 1):
        print(f"   {i}. Score: {source.score:.4f} - {source.text[:100]}...")

    print("\n" + "=" * 60)


❓ Pregunta: ¿Cuál fue la huella de carbono de Scotiabank Chile en 2023?
--------------------------------------------------
📚 Respuesta (con RAG):
5.981 tCO2e fue la huella de carbono de Scotiabank Chile en 2023, según el documento de resumen ASG 2023. Este dato se encuentra en la página 11 del archivo PDF, donde se menciona que durante el último año, la huella de carbono de Scotiabank Chile fue de 5.981 tCO2e. 

To elaborate further, the document states that "De acuerdo con nuestro cálculo, elaborado sobre la base del GHG Protocol, durante el último año, la huella de carbono de Scotiabank Chile fue de 5.981 tCO2e." This indicates that the calculation was based on the GHG Protocol, and the result for 2023 was 5.981 tons of CO2 equivalent (tCO2e).
The context also mentions that this figure represents the organization's efforts to monitor greenhouse gas emissions and reduce its environmental footprint. The document provides additional details about the organization's actions in managing 

### 💡 Ejercicio 2: Compara las respuestas

Haz tus propias preguntas y compara las respuestas con y sin RAG.

In [ ]:
# TODO: Escribe tu propia pregunta aquí
mi_pregunta = "¿Cuántos escolares participaron en programas de educación financiera?"

In [ ]:
# Sin RAG
print("\n🤖 Respuesta SIN RAG:")
print("-" * 50)
respuesta_sin_rag = llm.complete(mi_pregunta)
print(respuesta_sin_rag.text)


🤖 Respuesta SIN RAG:
--------------------------------------------------
 | Noticias de El Salvador - La Prensa Gráfica
El 2019, el 35% de los escolares participó en algún programa de educación financiera.
14 de Enero de 2020 - 16:27
Según un informe del Ministerio de Educación (Mined), el 35% de los escolares participó en algún programa de educación financiera durante el año 2019. El documento se titula "Evaluación de la implementación y efectividad de los Programas de Educación Financiera en las Instituciones Educativas".
El estudio fue realizado por el Mined y la Asociación de Educación Financiera (AEF) y se basa en una encuesta a 1,000 escolares de primaria y secundaria de todo el país.
El 35% de los escolares que participaron en algún programa de educación financiera durante el año 2019, lo hicieron en instituciones educativas públicas. El 38% de los escolares que participaron en algún programa de educación financiera lo hicieron en instituciones educativas privadas.
El 20% de los

In [ ]:
# Con RAG
print("\n📚 Respuesta CON RAG:")
print("-" * 50)
respuesta_con_rag = query_engine.query(mi_pregunta)
print(respuesta_con_rag.response)


📚 Respuesta CON RAG:
--------------------------------------------------
1.520 escolares participaron en los programas de educación financiera según el documento proporcionado. Estos programas incluyen el refuerzo de los programas de educación financiera para estudiantes y la ejecución de 18 actividades de voluntariado corporativo, que impactaron a más de 45 mil personas.
page_label: 13
file_path: Resumen_ASG_2023.pdf

Alcance
Programa “Creamos Futuro”
Alcance
91
COLEGIOS MUNICIPALIZADOS  DE 
LA REGIÓN METROPOLITANA.

page_label: 5
file_path: Resumen_ASG_2023.pdf

Diego Masola, 
EVP & Country Head Scotiabank Chile
esto debemos sumar la elaboración por parte del 
Consejo Asesor ASG de un Plan Director, que será 
la hoja de ruta que seguiremos en los próximos 
años en todo lo relacionado con la gestión y el 
cumplimiento ambiental.
Bajo el pilar Sociedad Inclusiva, en 2023, también 
impulsamos acciones significativas, como el 
reforzamiento de los programas de educación 
financiera para 

### 💡 Ejercicio 3: Más preguntas para explorar

Prueba con estas preguntas sobre el documento:

1. ¿Qué es Iniciativa Mujeres?
2. ¿Cuál es la brecha salarial en Scotiabank?
3. ¿Cuántos días tarda Scotiabank en pagar a proveedores?
4. ¿Qué certificaciones o reconocimientos recibió Scotiabank?
5. ¿Qué es el programa Silver Age?

## 11. 🧪 SECCIÓN OPCIONAL: Re-parsear con LlamaParse

**LlamaParse** es un servicio de Anthropic/LlamaIndex que usa modelos avanzados
para extraer mejor el contenido de PDFs, especialmente:
- Tablas
- Imágenes con texto
- Layouts complejos

⚠️ **Requiere API Key gratuita** (con límite de páginas diarias)

In [ ]:
# Configurar tu API Key de LlamaParse
# Obtén tu key gratis en: https://cloud.llamaindex.ai/
LLAMA_CLOUD_API_KEY = ""  # <-- Pega tu API key aquí

In [ ]:
if LLAMA_CLOUD_API_KEY:
    print("✓ API Key de LlamaParse configurada")
    USE_LLAMA_PARSE = True
else:
    print("⚠ Para usar LlamaParse, necesitas una API Key")
    print("  1. Ve a https://cloud.llamaindex.ai/")
    print("  2. Crea una cuenta gratuita")
    print("  3. Copia tu API Key y pégala arriba")
    USE_LLAMA_PARSE = False

✓ API Key de LlamaParse configurada


### 11.1 Instalar y Configurar LlamaParse

In [ ]:
if USE_LLAMA_PARSE:
    # Instalar llama-parse
    !pip install -q llama-parse

    from llama_parse import LlamaParse
    import nest_asyncio
    nest_asyncio.apply()  # Necesario para notebooks

    print("📄 Re-parseando documento con LlamaParse...")

    # Configurar parser
    parser_llama = LlamaParse(
        api_key=LLAMA_CLOUD_API_KEY,
        result_type="markdown",  # Mejor para tablas y estructura
        language="es",           # Español
        verbose=True
    )

    # Parsear el documento
    documents_llama = parser_llama.load_data(PDF_PATH)

    print(f"✓ Documento re-parseado: {len(documents_llama)} página(s)")

    # Mostrar ejemplo del contenido extraído
    print("\n📝 Ejemplo del contenido extraído con LlamaParse:")
    print("-" * 50)
    print(documents_llama[0].text[:1000])

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 397.2/397.2 kB 11.1 MB/s eta 0:00:00
📄 Re-parseando documento con LlamaParse...
Started parsing the file under job_id 14881ecf-82cd-4a8e-985f-eb6fccace5d6
✓ Documento re-parseado: 24 página(s)

📝 Ejemplo del contenido extraído con LlamaParse:
--------------------------------------------------

ACLA BU

# Soy Voluntari@

# Por un futuro sostenible

# El compromiso ASG de Scotiabank Chile en 2023

# Scotia ASG





In [ ]:
# Podemos guardar el documento para verlo mejor

# Guardar el documento parseado en markdown
with open("documento_parseado.md", "w", encoding="utf-8") as f:
    for doc in documents_llama:
        f.write(doc.text)
        f.write("\n\n---\n\n")  # Separador entre páginas

print("✓ Documento guardado en documento_parseado.md")

✓ Documento guardado en documento_parseado.md


### 11.2 Crear nuevo índice con el documento re-parseado

In [ ]:
if USE_LLAMA_PARSE:
    print("🔍 Creando nuevo índice con documento re-parseado...")

    # Dividir en chunks
    nodes_llama = parser.get_nodes_from_documents(documents_llama)
    print(f"✓ Documento dividido en {len(nodes_llama)} chunks")

    # Crear nuevo índice
    index_llama = VectorStoreIndex(nodes_llama)

    # Crear nuevo query engine
    query_engine_llama = index_llama.as_query_engine(similarity_top_k=3)

    print("✓ Nuevo índice creado con LlamaParse")

🔍 Creando nuevo índice con documento re-parseado...
✓ Documento dividido en 34 chunks
✓ Nuevo índice creado con LlamaParse


In [ ]:
# Crear retriever (buscador)
retriever2 = index_llama.as_retriever(
    similarity_top_k=3  # Retornar los 3 chunks más relevantes
)
print("=" * 60)
print("🔎 PROBANDO EL RETRIEVER")
print("=" * 60)

🔎 PROBANDO EL RETRIEVER


In [ ]:
retrieved2 = retriever2.retrieve('Directorio Paritario')
for i, node in enumerate(retrieved2, 1):
    print(f"\n🔹 Documento {i} (Score: {node.score:.4f}):")
    print(f"   {node.text[:400]}...")


🔹 Documento 1 (Score: 0.4576):
   A su vez, en diciembre, se presentó al Directorio la gestión y avances ASG durante el año 2023.

# LIDERAZGO Y GOBIERNO CORPORATIVO

El Gobierno Corporativo de Scotiabank Chile tiene como misión asegurar un adecuado funcionamiento de las operaciones de la organización y velar por la ejecución de las actividades de control interno y cumplimiento normativo. El Directorio del Banco, por su parte, tie...

🔹 Documento 2 (Score: 0.4555):
   # Gestión de proveedores con mirada de desarrollo mutuo

# Proveedores locales con contratos vigentes al cierre del año por región

| Arica y Parinacota | 3   | Maule                          | 10 |
| ------------------ | --- | ------------------------------ | -- |
| Tarapacá           | 4   | Ñuble                          | 5  |
| Antofagasta        | 10  | Biobío                         | 16 |
...

🔹 Documento 3 (Score: 0.4295):
   # Promovemos una conducta sustentada en la ética y el cumplimiento legal

# Total de pe

### 11.3 Comparar resultados: Parser normal vs LlamaParse

In [ ]:
if USE_LLAMA_PARSE:
    print("=" * 60)
    print("📊 COMPARANDO: Parser normal vs LlamaParse")
    print("=" * 60)

    preguntas_comparacion = [
        "¿Cuál fue la huella de carbono total en 2023 por alcance?",
        "¿Cuántos kilos de residuos se reciclaron?",
        "¿Cuáles organizaciones ganaron fondos de ScotiaInspira?",
    ]

    for pregunta in preguntas_comparacion:
        print(f"\n❓ Pregunta: {pregunta}")
        print("-" * 50)

        # Con parser normal
        print("\n📄 Respuesta (Parser normal):")
        resp_normal = query_engine.query(pregunta)
        print(resp_normal.response)

        # Con LlamaParse
        print("\n🚀 Respuesta (LlamaParse):")
        resp_llama = query_engine_llama.query(pregunta)
        print(resp_llama.response)

        print("\n" + "=" * 60)

📊 COMPARANDO: Parser normal vs LlamaParse

❓ Pregunta: ¿Cuál fue la huella de carbono total en 2023 por alcance?
--------------------------------------------------

📄 Respuesta (Parser normal):


KeyboardInterrupt: 

## 12. Resumen y Conclusiones

### ¿Qué aprendimos?

1. **Retriever**: Busca documentos semánticamente similares a la query
2. **LLM sin RAG**: Tiene conocimiento limitado y puede alucinar
3. **RAG**: Combina búsqueda + generación para respuestas precisas
4. **LlamaParse**: Mejora la extracción de contenido complejo (tablas, etc.)

### Flujo del RAG:
```
Query del usuario
       ↓
   Retriever (busca chunks relevantes)
       ↓
   Prompt = Query + Contexto recuperado
       ↓
   LLM genera respuesta
       ↓
   Respuesta final al usuario
```

### 💡 Ideas para mejorar tu RAG:

1. **Ajustar chunk_size**: Chunks más pequeños = más precisión, chunks más grandes = más contexto
2. **Aumentar similarity_top_k**: Más documentos = más contexto pero más ruido
3. **Usar re-ranking**: Ordenar los documentos recuperados por relevancia
4. **Mejorar el prompt**: Agregar instrucciones específicas al LLM
5. **Usar LlamaParse**: Para documentos con tablas o layouts complejos

## 13. Ejercicios Adicionales

### Ejercicio A: Modificar parámetros del retriever
Prueba diferentes valores de `similarity_top_k` (1, 3, 5, 10) y observa cómo cambian las respuestas.

### Ejercicio B: Cambiar el tamaño de chunks
Modifica `chunk_size` (256, 512, 1024) y compara la calidad del retrieval.

### Ejercicio C: Crear tu propio sistema de Q&A
Usa un documento diferente (puede ser un PDF de tu empresa, universidad, etc.) y crea un sistema RAG para responder preguntas sobre él.


In [ ]:
print("\n" + "=" * 60)
print("🎉 ¡Laboratorio completado!")
print("=" * 60)
print("""
Ahora sabes cómo:
✓ Cargar y procesar documentos PDF
✓ Crear embeddings y un índice vectorial
✓ Usar un retriever para buscar información
✓ Comparar respuestas con y sin RAG
✓ (Opcional) Usar LlamaParse para mejor extracción

¡Experimenta con tus propios documentos y preguntas!
""")